# InfiniSplat baseline from a PanoGS panorama

This notebook runs the **released InfiniSplat checkpoint** on one perspective view extracted from an equirectangular HDR panorama. It is a controlled baseline, not the panorama-adapted PanoGS model. All extracted crops share the panorama's original camera center; they are not independent photographs.

## Kaggle setup
1. Attach one of `brown_photostudio_02_2k.hdr`, `mirrored_hall_2k.hdr`, or `modern_bathroom_2k.hdr` as a private Kaggle input dataset.
2. Enable Internet and a GPU accelerator. This notebook stops early if CUDA is unavailable.
3. Run top to bottom. The setup cell creates an isolated Python 3.10 environment inside `/kaggle/working`; Kaggle's base Python stays unchanged. Inspect the crop before inference and change `YAW_DEG` if needed.
4. Download the output PLY from `/kaggle/working/infinisplat_baseline/`.

The HDR file is preserved. A tone-mapped RGB crop is made for the model, so the output PLY does not retain the panorama's full HDR radiance.

In [ ]:
from pathlib import Path
import json, math, os, subprocess, sys, time
import cv2
import numpy as np
from PIL import Image
import torch

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator, restart the session, and rerun.')

NAMES = ['brown_photostudio_02_2k.hdr', 'mirrored_hall_2k.hdr', 'modern_bathroom_2k.hdr']
found = {p.name: p for p in Path('/kaggle/input').rglob('*') if p.is_file() and p.name in NAMES}
if not found:
    raise FileNotFoundError('Attach one of the three HDR panorama files as a Kaggle input dataset.')
print('Available panoramas:', *found.keys(), sep='\n  ')
HDR_PATH = next(iter(found.values()))  # Change this to found['filename.hdr'] to select explicitly.
YAW_DEG = 0.0       # Center direction in panorama coordinates; adjust after preview.
FOV_DEG = 90.0      # Perspective crop field of view.
CROP_SIZE = None    # Set from the panorama's angular pixel density after reading the HDR.
print('Selected:', HDR_PATH)

## Project the panorama into a perspective camera view
This avoids feeding an equirectangular image to a model whose released inference path expects a perspective RGB image and camera intrinsics. The crop preview is the fair input for this first baseline.

In [ ]:
hdr_bgr = cv2.imread(str(HDR_PATH), cv2.IMREAD_UNCHANGED)
if hdr_bgr is None or hdr_bgr.ndim != 3:
    raise RuntimeError(f'Could not read an RGB HDR panorama: {HDR_PATH}')
hdr = cv2.cvtColor(hdr_bgr[..., :3], cv2.COLOR_BGR2RGB).astype(np.float32)
hdr = np.nan_to_num(hdr, nan=0.0, posinf=0.0, neginf=0.0).clip(0.0)
H, W = hdr.shape[:2]
if W / H < 1.8:
    raise ValueError(f'Expected an equirectangular 2:1 panorama, got {W}x{H}')
# A 2048x1024 ERP has about 512 source pixels across a 90-degree view. Avoid
# enlarging that view to 768px, which adds pixels but cannot add captured detail.
CROP_SIZE = max(256, int(round(W * FOV_DEG / 360.0)))

# Pinhole ray grid: camera looks along +Z; image Y points down.
f = (CROP_SIZE / 2.0) / math.tan(math.radians(FOV_DEG) / 2.0)
xx, yy = np.meshgrid(np.arange(CROP_SIZE, dtype=np.float32), np.arange(CROP_SIZE, dtype=np.float32))
local_x = (xx - (CROP_SIZE - 1) / 2.0) / f
local_y = -((yy - (CROP_SIZE - 1) / 2.0) / f)
yaw = math.radians(YAW_DEG)
dx = math.cos(yaw) * local_x + math.sin(yaw)
dy = local_y
dz = -math.sin(yaw) * local_x + math.cos(yaw)
theta = np.arctan2(dx, dz)
phi = np.arctan2(dy, np.sqrt(dx * dx + dz * dz))
map_x = ((theta / (2 * np.pi) + 0.5) * W - 0.5) % W
map_y = np.clip((0.5 - phi / np.pi) * H - 0.5, 0, H - 1)
# Triple the panorama horizontally so bilinear samples wrap cleanly across the seam.
hdr_wrapped = np.concatenate([hdr, hdr, hdr], axis=1)
crop_hdr = cv2.remap(hdr_wrapped, (map_x + W).astype(np.float32), map_y.astype(np.float32), cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

# Robust display transform: compress highlights without clipping the HDR source.
luma = np.maximum(crop_hdr @ np.array([0.2126, 0.7152, 0.0722], np.float32), 1e-6)
log_avg = float(np.exp(np.mean(np.log(luma))))
exposed = crop_hdr * (0.18 / max(log_avg, 1e-6))
ldr = exposed / (1.0 + exposed)
srgb = np.where(ldr <= 0.0031308, 12.92 * ldr, 1.055 * np.power(ldr, 1/2.4) - 0.055)
crop = Image.fromarray(np.uint8(np.clip(srgb, 0, 1) * 255))
WORK = Path('/kaggle/working/infinisplat_baseline')
WORK.mkdir(parents=True, exist_ok=True)
INPUT_IMAGE = WORK / 'panorama_perspective_crop.png'
crop.save(INPUT_IMAGE)
print('HDR:', W, 'x', H, '| crop:', crop.size, '| yaw:', YAW_DEG, '| FOV:', FOV_DEG)
display(crop)

## Install the released implementation
This uses the authors' repository and released checkpoint. The official setup targets Python 3.10 and CUDA 12.8. We install those into a separate virtual environment, so this does not replace Kaggle's base Python or PyTorch. Internet must be enabled.

In [ ]:
REPO = Path('/kaggle/working/InfiniSplat')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/zju3dv/InfiniSplat.git', str(REPO)], check=True)
# Create a dedicated Python 3.10 runtime; do not change Kaggle's active environment.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
import shutil
UV = shutil.which('uv')
if not UV:
    raise RuntimeError('uv was installed but is not on PATH; restart the Kaggle session and rerun this cell.')
subprocess.run([UV, 'python', 'install', '3.10'], check=True)
MODEL_ENV = Path('/kaggle/working/infinisplat-py310')
# The --clear flag also recovers cleanly if an earlier setup attempt stopped midway.
subprocess.run([UV, 'venv', '--clear', '--python', '3.10', str(MODEL_ENV)], check=True)
MODEL_PY = MODEL_ENV / 'bin' / 'python'
# Versions match the authors' CUDA 12.8 installation guide.
subprocess.run([UV, 'pip', 'install', '--python', str(MODEL_PY), '--index-url', 'https://download.pytorch.org/whl/cu128', 'torch==2.9.0', 'torchvision==0.24.0', 'xformers==0.0.33.post1'], check=True)
# Video/HTML viewer extras are unnecessary for PLY export, so omit their large optional dependencies.
req_lines = (REPO / 'requirements.txt').read_text(encoding='utf-8').splitlines()
req_lines = [line for line in req_lines if not line.strip().startswith(('gradio==', 'nodejs-wheel=='))]
REQ_FILE = WORK / 'infinisplat-requirements.txt'
REQ_FILE.write_text('\n'.join(req_lines) + '\n', encoding='utf-8')
subprocess.run([UV, 'pip', 'install', '--python', str(MODEL_PY), '-r', str(REQ_FILE)], check=True)
subprocess.run([str(MODEL_PY), '-c', "import sys, torch; print('Model Python:', sys.version.split()[0]); print('Model PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| available:', torch.cuda.is_available())"], check=True)
print('Repository:', REPO)
print('Isolated environment:', MODEL_ENV)

## Download the released RGB checkpoint
The code repository does not include model weights. This downloads only the RGB checkpoint (about 3.14 GB) from the authors' Hugging Face repository; the larger LiDAR checkpoint is not needed for this test.

In [ ]:
MODEL_CHECKPOINT = REPO / 'checkpoints' / 'infinisplat_rgb.ckpt'
MODEL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not MODEL_CHECKPOINT.is_file() or MODEL_CHECKPOINT.stat().st_size < 2 * 1024**3:
    download_code = (
        "from huggingface_hub import hf_hub_download; "
        f"hf_hub_download(repo_id='PLUS-WAVE/InfiniSplat', "
        f"filename='checkpoints/infinisplat_rgb.ckpt', local_dir={str(REPO)!r})"
    )
    subprocess.run([str(MODEL_PY), '-c', download_code], check=True)
if not MODEL_CHECKPOINT.is_file() or MODEL_CHECKPOINT.stat().st_size < 2 * 1024**3:
    raise FileNotFoundError(f'RGB checkpoint download is missing or incomplete: {MODEL_CHECKPOINT}')
print(f'Checkpoint ready: {MODEL_CHECKPOINT} ({MODEL_CHECKPOINT.stat().st_size / 1024**3:.2f} GiB)')

## Infer and collect the PLY
Inference uses RGB-only mode. The focal length is passed explicitly because this crop has known synthetic pinhole intrinsics. Video and HTML exports are disabled to keep the baseline focused on the Gaussian PLY.

In [ ]:
focal_px = (CROP_SIZE / 2.0) / math.tan(math.radians(FOV_DEG) / 2.0)
cuda_check = subprocess.run([str(MODEL_PY), '-c', "import torch; assert torch.cuda.is_available(), 'CUDA unavailable in isolated InfiniSplat environment'; print(torch.cuda.get_device_name(0))"], check=True, capture_output=True, text=True)
infer_cmd = [str(MODEL_PY), '-m', 'src.demo.infer_batch_images', '--input', str(INPUT_IMAGE), '--output-dir', str(WORK / 'model_output'), '--checkpoint', str(MODEL_CHECKPOINT), '--focal-px', str(focal_px), '--no-video', '--no-export-html']
print('GPU:', cuda_check.stdout.strip())
print('Command:', ' '.join(infer_cmd))
started = time.time()
# Kaggle exports its notebook-only inline backend into child processes. The
# isolated model environment has no inline backend, so force a headless one.
model_env = os.environ.copy()
model_env['MPLBACKEND'] = 'Agg'
subprocess.run(infer_cmd, cwd=REPO, check=True, env=model_env)
ply_files = list((WORK / 'model_output').rglob('*.ply'))
if not ply_files:
    raise FileNotFoundError('InfiniSplat finished without a PLY; inspect the inference output above.')
OUTPUT_PLY = WORK / 'infinisplat_perspective_baseline.ply'
if ply_files[0] != OUTPUT_PLY:
    import shutil
    shutil.copy2(ply_files[0], OUTPUT_PLY)
print(f'PLY: {OUTPUT_PLY} ({OUTPUT_PLY.stat().st_size / 1024**2:.1f} MiB)')
print(f'Inference elapsed: {time.time() - started:.1f} s')

In [ ]:
manifest = {
    'source_hdr': HDR_PATH.name,
    'input_projection': 'equirectangular panorama to perspective pinhole crop',
    'yaw_degrees': YAW_DEG,
    'fov_degrees': FOV_DEG,
    'crop_size': CROP_SIZE,
    'model': 'PLUS-WAVE/InfiniSplat released RGB checkpoint',
    'output': OUTPUT_PLY.name,
    'limitations': 'Single panorama center; inferred geometry; tone-mapped RGB input; not a full 360 panorama reconstruction.'
}
(WORK / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
print('Download the PLY and manifest from the Kaggle output files, then compare in the PanoGS viewer from the crop direction.')